# Chapter 24 Coding Exam — Tree Ensembles, OOB Tuning & Shapley Interpretability (100 Points)

**Book Reference:** Chapter 24, Section 24.7 (Chapter 24 Coding Exam)  
**Suggested Time:** 45–55 minutes (Open Book, Closed Solution)

| Task | Tree Ensemble & Interpretability Challenge | Points |
| :--- | :--- | :---: |
| **Task 24.1** | `exact_shapley_3features(predict_fn, x_row, baseline_row)` (Exact 6-permutation Shapley values & Efficiency Axiom) | 30 |
| **Task 24.2** | `tune_random_forest_oob(X_train, y_train, max_depths, seed=42)` (Out-of-Bag depth selection & audit table) | 35 |
| **Task 24.3** | `public_health_tree_benchmark(mch_df, seed=42)` (MCH stunting cleaning, 3-family benchmark & winning-model permutation top driver) | 35 |
| **TOTAL** | | **100** |

In [ ]:
# @title Step 0: Click Play (▶) to Activate Chapter 24 Exam Grader
!pip install -q sharukh-tutor
import itertools
import numpy as np
import pandas as pd
from sklearn.ensemble import (
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.inspection import permutation_importance
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
import sharukh_tutor as tutor

tutor.activate(chapter="ch24", notebook="coding_exam")


---
## Exam Task 24.1 (30 Points): `exact_shapley_3features(predict_fn, x_row, baseline_row)`

Compute exact Shapley attributions for a 3-feature sample `x_row` relative to `baseline_row`, where `predict_fn(v)` takes a 1D 3-element vector and returns a scalar float:
1. Compute `base_value = predict_fn(baseline_row)` and `prediction = predict_fn(x_row)`.
2. Iterate over all $3! = 6$ permutations of `[0, 1, 2]` (`itertools.permutations([0, 1, 2])`). For each order, start from a fresh copy of `baseline_row`, reveal `x_row[idx]` one feature at a time, and record each feature's marginal change `curr - prev`.
3. Average each feature's marginal contributions over the 6 permutations so the Efficiency Axiom $\phi_0 + \sum_{i=1}^{3} \phi_i = f(x)$ holds.
4. Return `{"base_value": round4, "shap_values": [round4, round4, round4], "prediction": round4}` without mutating inputs.

In [ ]:
def exact_shapley_3features(predict_fn, x_row, baseline_row):
    """Compute exact 3-feature Shapley values over all 6 permutations."""
    # STUB: raise NotImplementedError("Replace with your solution")
    ### BEGIN SOLUTION
    x = np.asarray(x_row, dtype=float)
    base = np.asarray(baseline_row, dtype=float)
    base_val = float(predict_fn(base))
    pred_val = float(predict_fn(x))
    contribs = np.zeros(3, dtype=float)
    perms = list(itertools.permutations([0, 1, 2]))
    for perm in perms:
        state = base.copy()
        prev = base_val
        for idx in perm:
            state[idx] = x[idx]
            curr = float(predict_fn(state))
            contribs[idx] += curr - prev
            prev = curr
    shap_vals = contribs / len(perms)
    return {
        "base_value": round(base_val, 4),
        "shap_values": [round(float(v), 4) for v in shap_vals],
        "prediction": round(pred_val, 4),
    }
    ### END SOLUTION


---
## Exam Task 24.2 (35 Points): `tune_random_forest_oob(X_train, y_train, max_depths, seed=42)`

Tune Random Forest depth using Out-of-Bag (OOB) accuracy without a separate validation split:
1. For each `d` in `max_depths`, fit `RandomForestClassifier(n_estimators=120, max_depth=d, oob_score=True, random_state=seed)` on `(X_train, y_train)`.
2. Build `oob_table` as a `pd.DataFrame` with columns `["max_depth", "oob_score"]` rounded to **4 decimal places**.
3. Select the depth with the highest `oob_score` and return `{"best_depth": int, "best_oob_score": round4, "oob_table": oob_table}`.

In [ ]:
def tune_random_forest_oob(X_train, y_train, max_depths, seed=42):
    """Select best RandomForest max_depth using Out-of-Bag accuracy."""
    # STUB: raise NotImplementedError("Replace with your solution")
    ### BEGIN SOLUTION
    rows = []
    best_depth = int(max_depths[0])
    best_oob = -1.0
    for d in max_depths:
        rf = RandomForestClassifier(
            n_estimators=120, max_depth=d,
            oob_score=True, random_state=seed,
        ).fit(X_train, y_train)
        score = float(rf.oob_score_)
        rows.append({"max_depth": int(d), "oob_score": round(score, 4)})
        if score > best_oob:
            best_oob = score
            best_depth = int(d)
    return {
        "best_depth": best_depth,
        "best_oob_score": round(best_oob, 4),
        "oob_table": pd.DataFrame(rows),
    }
    ### END SOLUTION


---
## Exam Task 24.3 (35 Points): `public_health_tree_benchmark(mch_df, seed=42)`

Build a complete child-stunting tree benchmark on `mch_survey_sample` without mutating `mch_df`:
1. Copy `mch_df`, standardize `residence` (`str.strip().str.title()`), replace sentinel `anc_visits == 999` with `np.nan`, compute `is_urban = (residence == "Urban").astype(int)`, and map `wealth_quintile` (`Poorest: 1 .. Richest: 5`) to `wealth_rank`.
2. Drop rows with missing values across `["maternal_age_years", "anc_visits", "maternal_hb_g_dl", "is_urban", "wealth_rank", "child_stunted"]`.
3. Run a stratified 75/25 split (`test_size=0.25, stratify=y, random_state=seed`).
4. Fit `DecisionTree` (`max_depth=5`), `RandomForest` (`n_estimators=100, max_depth=6`), and `HistGradientBoosting` (`max_depth=4`), all with `random_state=seed`. Build `leaderboard` (`["model", "roc_auc", "pr_auc"]`) sorted by `roc_auc` descending, rounded to **4 decimals**.
5. Run held-out `permutation_importance` (`scoring="roc_auc", n_repeats=10, random_state=seed`) on the winning `best_model` to identify `top_feature`, and return `{"leaderboard": leaderboard, "best_model": best_model, "top_feature": top_feature}`.

In [ ]:
def public_health_tree_benchmark(mch_df, seed=42):
    """Benchmark 3 tree families on MCH stunting and rank top driver."""
    # STUB: raise NotImplementedError("Replace with your solution")
    ### BEGIN SOLUTION
    work = mch_df.copy()
    work["residence"] = (
        work["residence"].astype(str).str.strip().str.title()
    )
    work["anc_visits"] = work["anc_visits"].replace(999, np.nan)
    work["is_urban"] = (work["residence"] == "Urban").astype(int)
    w_map = {
        "Poorest": 1, "Poorer": 2, "Middle": 3,
        "Richer": 4, "Richest": 5,
    }
    work["wealth_rank"] = (
        work["wealth_quintile"].astype(str).str.strip().str.title()
        .map(w_map)
    )
    feats = [
        "maternal_age_years", "anc_visits", "maternal_hb_g_dl",
        "is_urban", "wealth_rank",
    ]
    clean = work[feats + ["child_stunted"]].dropna()
    X = clean[feats]
    y = clean["child_stunted"].astype(int)
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.25, stratify=y, random_state=seed
    )
    specs = [
        ("DecisionTree", DecisionTreeClassifier(
            max_depth=5, random_state=seed)),
        ("RandomForest", RandomForestClassifier(
            n_estimators=100, max_depth=6, random_state=seed)),
        ("HistGradientBoosting", HistGradientBoostingClassifier(
            max_depth=4, random_state=seed)),
    ]
    rows, fitted = [], {}
    for name, clf in specs:
        clf.fit(X_tr, y_tr)
        proba = clf.predict_proba(X_te)[:, 1]
        rows.append({
            "model": name,
            "roc_auc": round(float(roc_auc_score(y_te, proba)), 4),
            "pr_auc": round(
                float(average_precision_score(y_te, proba)), 4
            ),
        })
        fitted[name] = clf
    leaderboard = (
        pd.DataFrame(rows)
        .sort_values("roc_auc", ascending=False)
        .reset_index(drop=True)
    )
    best_model = str(leaderboard.iloc[0]["model"])
    pi = permutation_importance(
        fitted[best_model], X_te, y_te, scoring="roc_auc",
        n_repeats=10, random_state=seed,
    )
    top_feature = feats[int(np.argmax(pi.importances_mean))]
    return {
        "leaderboard": leaderboard,
        "best_model": best_model,
        "top_feature": top_feature,
    }
    ### END SOLUTION


In [ ]:
exam_submissions = {
    "ch24_exam_01": exact_shapley_3features,
    "ch24_exam_02": tune_random_forest_oob,
    "ch24_exam_03": public_health_tree_benchmark,
}

tutor.check_exam("ch24", exam_submissions)


---
## Chapter 24 Exam Debugging Post-Mortem (Solution Notebook Only)

- **Task 24.1 (Order Dependence & Interaction Splitting):** On nonlinear functions with interactions like $4 x_0 x_2$, evaluating only one ordering assigns 100% of the interaction to whichever feature arrived second! Averaging across all $3! = 6$ permutations guarantees fair symmetric credit (`+2.0` each) and enforces the Efficiency Axiom $\phi_0 + \sum \phi_i = f(x)$.
- **Task 24.2 (OOB Requires `bootstrap=True`):** Out-of-Bag validation relies on each bootstrap sample omitting ~36.8% of training rows (`(1 - 1/n)^n -> 1/e`). Setting `bootstrap=False` raises `ValueError` because there are no out-of-bag rows left to score.
- **Task 24.3 (Empirical Humility on Small Surveys):** Always clean sentinel `999` and map ordinal wealth quintiles cleanly before splitting, and never assume the most complex ensemble automatically wins on small $n = 470$ survey cohorts — benchmark all three families and verify the top risk driver with held-out permutation importance!